# Notebook 03_D Moodle Feature Ablation
Compare the two finalist model configurations using:

- **Full:** all 24 predictors
- **No Moodle:** all Moodle predictors removed

The same training data and five fold stratified cross validation are used.  
The held out test set is not used in this ablation.

Main comparison: **F1, PR AUC, Recall**.

In [5]:
from pathlib import Path
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data" / "processed" / "ml_ready"
OUTPUT_DIR = PROJECT_DIR / "results" / "ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_pickle(DATA_DIR / "X_train_raw.pkl")
y_train = pd.read_pickle(DATA_DIR / "y_train.pkl")

print("Training shape:", X_train.shape)


Training shape: (3008, 24)


In [9]:
MOODLE_FEATURES = [
    "enrolled_course_count",
    "accessed_course_count",
    "course_access_rate",
    "total_course_event_clicks",
    "active_days_numeric",
    "active_days_rate",
    "zero_activity_days",
    "largest_inactivity_days",
    "learning_material_events_ordinal",
    "assessment_interaction_events_ordinal",
]

CATEGORICAL = [
    "programme_or_school",
    "year_level",
    "previous_academic_standing",
]

FULL_FEATURES = list(X_train.columns)
NO_MOODLE_FEATURES = [f for f in FULL_FEATURES if f not in MOODLE_FEATURES]

assert len(FULL_FEATURES) == 24
assert len(NO_MOODLE_FEATURES) == 14

def make_pipeline(model, features):
    cat = [f for f in CATEGORICAL if f in features]
    num = [f for f in features if f not in cat]

    preprocessor = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), cat),
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        max_features=0.7,
        max_depth=12,
        class_weight={0: 1, 1: 3},
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=1,
        subsample=1.0,
        colsample_bytree=1.0,
        scale_pos_weight=3.03,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
scoring = {"f1": "f1", "pr_auc": "average_precision", "recall": "recall"}


print("Full features:", len(FULL_FEATURES))
print("No Moodle features:", len(NO_MOODLE_FEATURES))
print("Removed:", MOODLE_FEATURES)



Full features: 24
No Moodle features: 14
Removed: ['enrolled_course_count', 'accessed_course_count', 'course_access_rate', 'total_course_event_clicks', 'active_days_numeric', 'active_days_rate', 'zero_activity_days', 'largest_inactivity_days', 'learning_material_events_ordinal', 'assessment_interaction_events_ordinal']


In [7]:
rows = []

for model_name, model in models.items():
    for feature_set, features in [
        ("Full", FULL_FEATURES),
        ("No Moodle", NO_MOODLE_FEATURES),
    ]:
        scores = cross_validate(
            make_pipeline(model, features),
            X_train[features],
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
        )

        rows.append({
            "Model": model_name,
            "Feature Set": feature_set,
            "Predictors": len(features),
            "F1": scores["test_f1"].mean(),
            "PR AUC": scores["test_pr_auc"].mean(),
            "Recall": scores["test_recall"].mean(),
        })

results = pd.DataFrame(rows)
display(results.round(3))


,Model,Feature Set,Predictors,F1,PR AUC,Recall
0,Random Forest,Full,24,0.813,0.888,0.899
1,Random Forest,No Moodle,14,0.821,0.895,0.904
2,XGBoost,Full,24,0.812,0.889,0.900
3,XGBoost,No Moodle,14,0.813,0.893,0.901


In [8]:
# Difference after removing Moodle: No Moodle - Full
comparison = results.pivot(index="Model", columns="Feature Set", values=["F1", "PR AUC", "Recall"])

delta = pd.DataFrame({
    "Model": comparison.index,
    "Δ F1": comparison[("F1", "No Moodle")] - comparison[("F1", "Full")],
    "Δ PR AUC": comparison[("PR AUC", "No Moodle")] - comparison[("PR AUC", "Full")],
    "Δ Recall": comparison[("Recall", "No Moodle")] - comparison[("Recall", "Full")],
}).reset_index(drop=True)

display(delta.round(4))

results.to_csv(OUTPUT_DIR / "moodle_feature_ablation.csv", index=False)
delta.to_csv(OUTPUT_DIR / "moodle_feature_ablation_delta.csv", index=False)


,Model,Δ F1,Δ PR AUC,Δ Recall
0,Random Forest,0.0085,0.0073,0.0041
1,XGBoost,0.0016,0.0045,0.0013
